# Lezione 15 (Chiusura) — Progetto integrativo: Extract → Transform → Load

## Obiettivi della lezione

Al termine di questa lezione sarai in grado di:

- Spiegare perché nessun progetto reale di dati usa un unico strumento, ma una **sequenza**: estrazione, trasformazione e caricamento (ETL).
- Riconoscere, in un problema reale, quale capitolo del corso apporta ogni pezzo della soluzione.
- Riprodurre da capo a fondo il progetto "Data Engineer junior": estrarre una tabella di PIL per paese, ripulirla e trasformarla, ed esportarla in CSV.
- Costruire codice di scraping **robusto**, con un piano B per quando la fonte esterna non risponde.

Questa è l'ultima lezione del corso. Non imparerai sintassi nuova: vedrai come tutto ciò che già sai (Blocco 1: Python di base; Blocco 2: file, NumPy, Pandas; Blocco 3: API e scraping) si incastra in un unico flusso di lavoro.

## Dalla teoria alla pratica

Ripassa per un attimo tutto quello che già sai fare:

- **Blocco 1** (Lezioni 1-8): variabili, funzioni, le tue classi (OOP), gestione degli errori.
- **Blocco 2** (Lezioni 9-11): file, calcolo numerico con NumPy, pulizia ed esplorazione di tabelle con Pandas.
- **Blocco 3** (Lezioni 12-14): richiedere dati a un'API, estrarre dati direttamente da una pagina web (scraping).

Ogni lezione è sembrata un pezzo isolato mentre la seguivi. Oggi vedrai che questi pezzi si incastrano: un problema reale quasi mai si risolve con un unico strumento, ma incatenandone diversi seguendo il ciclo **Extract → Transform → Load (ETL)**.

## L'incarico

Un'azienda internazionale che vuole espandersi in diversi paesi ti assume come Data Engineer junior. Il tuo compito: costruire uno script che estragga la lista delle **10 economie più grandi del mondo**, ordinate dal PIL più alto al più basso in miliardi di dollari (billion USD, arrotondato a 2 decimali), secondo i dati pubblicati dal Fondo Monetario Internazionale (FMI).

I dati si trovano in una tabella dentro una pagina di Wikipedia archiviata dalla Wayback Machine (perché il risultato sia riproducibile):

```
https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29
```

Questo è il lavoro reale di un Data Engineer: la fonte quasi mai arriva nel formato di cui hai bisogno, e non è nemmeno garantito che la fonte esterna risponda sempre. Per questo, nel codice di questa lezione vedrai un `try/except`: **tentiamo lo scraping reale** contro quella URL e, se fallisce o impiega troppo tempo (una rete instabile, la Wayback Machine non disponibile, un firewall dell'aula), proseguiamo con una **tabella di riserva** fittizia ma realistica, così che la lezione non si blocchi mai per un problema di connettività. Tutto il resto del flusso (pulizia, trasformazione, esportazione) è identico sia che si usi la tabella reale sia quella di riserva.

## Passo 0 — Setup: importare le librerie

Come in qualsiasi notebook di Pandas/NumPy (Lezioni 10 e 11), importiamo tutto all'inizio.

In [ ]:
import numpy as np
import pandas as pd

# Sopprimiamo i warning per tenere pulito l'output della lezione
import warnings
warnings.filterwarnings('ignore')

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## Passo 1 — Estrarre (Extract): da una pagina web a un DataFrame

*Strumento: Lezione 14, Web Scraping.*

Nel capitolo del Web Scraping hai visto `BeautifulSoup` cercare i tag a mano. Pandas ti risolve il lavoro pesante con un'unica funzione: `pd.read_html()` percorre l'HTML di una URL, trova **tutte** le tabelle della pagina e te le restituisce già convertite in una lista di DataFrame.

Nota bene: `pd.read_html()` **è** scraping. La differenza rispetto a quanto visto nella lezione precedente è che Pandas ti risparmia di scrivere il percorso dell'albero HTML perché lo schema "tabella di una pagina web → tabella di dati" è così comune che è già risolto.

**Nota sulla robustezza:** lo scraping in tempo reale dipende dal fatto che la Wayback Machine sia disponibile e risponda velocemente in questo momento. Perché la lezione non si fermi per questo, proviamo lo scraping reale con un `try/except` e, se fallisce, proseguiamo con una tabella di riserva. **Se hai una connessione stabile e vuoi riprodurre il progetto originale così com'è**, puoi eseguire soltanto il blocco del `try` contro la URL reale indicata sopra.

In [ ]:
URL = "https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29"

df = None

try:
    # --- Tentativo reale di scraping (può richiedere tempo o fallire senza internet stabile) ---
    # pd.read_html() restituisce una LISTA di DataFrame, uno per ogni tabella che trova nella pagina
    tables = pd.read_html(URL)

    # La tabella che ci serve (paese + PIL del FMI) è la terza della pagina
    df = tables[3]
    fuente = "scraping reale (Wayback Machine)"

except Exception as error:
    # --- Piano B: se lo scraping fallisce (senza internet, timeout, la pagina è cambiata), usiamo
    # una tabella di riserva fittizia ma realistica, con la STESSA struttura della tabella
    # originale: nome del paese nella colonna 0 e PIL del FMI in milioni USD nella colonna 2.
    print(f"Non è stato possibile fare scraping in diretta ({type(error).__name__}: {error}).")
    print("Proseguiamo con una tabella di riserva per non fermare la lezione.\n")

    datos_respaldo = {
        0: ['World', 'United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom'],
        1: ['-', '-', '-', '-', '-', '-', '-'],  # colonna che non usiamo (es. regione/anno)
        2: [105000000, 26950000, 17700000, 4430000, 4230000, 3730000, 3330000],  # PIL (FMI) in milioni USD
    }
    df = pd.DataFrame(datos_respaldo)
    fuente = "tabella di riserva (offline)"

print("Fonte dei dati usata:", fuente)
df.head()

## Passo 2 — Trasformare, parte 1: dare forma alla tabella

*Strumento: Lezione 11, Pandas (`.iloc`).*

La tabella reale di Wikipedia porta colonne con nomi composti e intestazioni raggruppate. Invece di lottare con quei nomi, sostituiamo le intestazioni con la loro posizione numerica: così possiamo riferirci a qualsiasi colonna tramite il suo indice, indipendentemente da come si chiamava originariamente. Questo funziona allo stesso modo sia sulla tabella reale sia su quella di riserva, perché entrambe hanno la stessa struttura di colonne.

In [ ]:
# Sostituiamo le intestazioni testuali con numeri di colonna (0, 1, 2, ...)
df.columns = range(df.shape[1])

# Teniamo solo la colonna 0 (nome del paese) e la colonna 2 (PIL secondo il FMI)
df = df[[0, 2]]

# Con .iloc (Lezione 11) selezioniamo le righe da 1 a 10: la top 10 delle economie del mondo
# (la riga 0 è "World"/totale, non ci interessa per la classifica)
df = df.iloc[1:11, :]

# Rinominiamo le colonne perché il risultato sia leggibile
df.columns = ['Country', 'GDP (Million USD)']

df

## Passo 3 — Trasformare, parte 2: convertire le unità con NumPy

*Strumento: Lezione 10, NumPy + Lezione 11, Pandas.*

Il FMI riporta il PIL in milioni di dollari, ma l'incarico chiede miliardi di dollari (billion USD). Prima di poter dividere dobbiamo assicurarci che la colonna sia numerica — arrivando da una tabella HTML (o da un dizionario con interi, nel nostro caso di riserva), conviene forzare il tipo con `.astype()` perché il codice funzioni allo stesso modo in entrambi i casi.

In [ ]:
# Ci assicuriamo che la colonna sia numerica prima di operarci
df['GDP (Million USD)'] = df['GDP (Million USD)'].astype(int)

# Operazione aritmetica sull'intera colonna: da milioni a miliardi.
# Questo è vettorizzato (senza ciclo for) perché una colonna di Pandas si appoggia su NumPy.
df[['GDP (Million USD)']] = df[['GDP (Million USD)']] / 1000

# np.round() (Lezione 10) arrotonda ogni valore a 2 decimali, come richiesto dall'incarico
df[['GDP (Million USD)']] = np.round(df[['GDP (Million USD)']], 2)

# Ora che l'unità è cambiata, anche il nome della colonna deve rifletterlo
df = df.rename(columns={'GDP (Million USD)': 'GDP (Billion USD)'})

df

## Passo 4 — Caricare (Load): salvare il risultato in un CSV

*Strumento: Lezione 9, File.*

L'ultimo passo dell'ETL è lasciare il risultato in un posto dove un'altra persona (o tu stesso, la settimana prossima) possa riutilizzarlo senza ripetere tutto il lavoro di scraping e pulizia. Con quello che già sai dalla lezione sui file, questa è una sola riga.

In [ ]:
df.to_csv('./Largest_economies.csv')
print("File 'Largest_economies.csv' generato correttamente.")

# Verifichiamo rileggendo il file
pd.read_csv('./Largest_economies.csv', index_col=0)

## Il flusso completo, da capo a fondo

Ecco come appare il progetto completo se lo scriviamo come un unico script, incatenando i quattro passi (con la stessa logica di `try/except` perché sia robusto in qualsiasi ambiente):

```python
import numpy as np
import pandas as pd

# --- EXTRACT (Lezione 14: Web Scraping) ---
URL = "https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29"
try:
    tables = pd.read_html(URL)
    df = tables[3]
except Exception:
    datos_respaldo = {
        0: ['World', 'United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom'],
        1: ['-', '-', '-', '-', '-', '-', '-'],
        2: [105000000, 26950000, 17700000, 4430000, 4230000, 3730000, 3330000],
    }
    df = pd.DataFrame(datos_respaldo)

# --- TRANSFORM, parte 1: dare forma alla tabella (Lezione 11: Pandas) ---
df.columns = range(df.shape[1])
df = df[[0, 2]]
df = df.iloc[1:11, :]
df.columns = ['Country', 'GDP (Million USD)']

# --- TRANSFORM, parte 2: convertire le unità (Lezione 10: NumPy) ---
df['GDP (Million USD)'] = df['GDP (Million USD)'].astype(int)
df[['GDP (Million USD)']] = df[['GDP (Million USD)']] / 1000
df[['GDP (Million USD)']] = np.round(df[['GDP (Million USD)']], 2)
df = df.rename(columns={'GDP (Million USD)': 'GDP (Billion USD)'})

# --- LOAD (Lezione 9: File) ---
df.to_csv('./Largest_economies.csv')
```

Dieci righe di logica reale, appoggiate su quattro lezioni diverse del corso. Questo è lo schema che ritroverai continuamente in qualsiasi lavoro con i dati: nessun passo è difficile da solo, l'abilità sta nel sapere in che ordine incatenarli e quale strumento usare in ciascuno.

## Altri progetti del corso che seguivano già questo stesso schema

Senza chiamarlo "ETL" in quel momento, hai già risolto lo stesso ciclo diverse volte:

- **Statistiche NBA via API** (Lezione 13): chiedere all'API (`extract`) → tabulare in un DataFrame e filtrare per squadra (`transform`) → confrontare medie e graficare.
- **TopSellingAlbums** (Lezione 11): caricare un CSV (`extract`) → selezionare colonne con `df['Length']` vs. `df[['Length']]` e righe con `.iloc`/`.loc` (`transform`) → la stessa coppia di strumenti che hai usato per ritagliare la tabella del PIL.
- **Dataset del diabete** (Lezione 11): esplorare con `.head()`, `.shape()`, `.info()`, `.describe()` prima di trarre conclusioni — il primo passo di qualsiasi `transform` su un dataset nuovo.
- **TextAnalyzer** (Lezione 7, OOP): una classe che organizza pulizia del testo e conteggio di frequenze — la stessa idea di incatenare passi, applicata con le tue classi invece che con Pandas.

## Riepilogo della lezione

- Nessun progetto reale usa un unico strumento: quasi sempre combinerai estrazione (scraping o API), trasformazione (Pandas e NumPy) e caricamento (file) nella stessa sequenza.
- Il ciclo **Extract → Transform → Load** è letteralmente l'ordine in cui hai risolto il progetto del PIL, passo per passo.
- `pd.read_html()` è la tua prima opzione quando i dati che ti servono sono già in una tabella HTML.
- `.iloc` (e `.loc`) sono gli strumenti che usi costantemente per ritagliare esattamente le righe e le colonne di cui hai bisogno.
- Le operazioni aritmetiche su intere colonne di un DataFrame (come convertire milioni in miliardi) funzionano perché Pandas si appoggia sul comportamento vettorizzato di NumPy.
- Uno scraper "di produzione" non assume mai che la fonte esterna risponda sempre: lo schema `try/except` con un piano B (o almeno un buon messaggio d'errore) fa parte del lavoro, non è un extra.
- Con questo si chiude il corso: la prossima abilità non è più imparare altra sintassi Python, ma esercitare il criterio di scegliere quale strumento usare in ogni passo di un problema nuovo.

### Per continuare a esercitarti da solo

Scegli un dataset pubblico che ti interessi davvero e percorri il ciclo completo: **estrai** (`pd.read_csv()`, `pd.read_html()`, un'API con `requests`...), **trasforma** (seleziona colonne/righe, correggi i tipi con `.astype()`, applica qualche operazione con Pandas o NumPy) e **carica** il risultato in un file. Non c'è una soluzione unica — l'obiettivo è che tu senta che il flusso completo è ormai tuo.